# Synthetic anomaly graphs

Blue points are normal windows; red points are windows for the selected injected anomaly type.


In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px


In [14]:
DATA_PATH = Path.cwd() / 'ampds_with_injected_anomalies.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Cannot find {DATA_PATH}')

data = pd.read_csv(DATA_PATH, parse_dates=['timestamp', 'window_id'])
data = data.sort_values('timestamp')
data['anomaly_type'] = data['anomaly_type'].fillna('normal')

print(f'Loaded {len(data):,} labeled minute-level samples')
display(data['anomaly_type'].value_counts())


Loaded 87,840 labeled minute-level samples


anomaly_type
normal                              76470
stuck_appliance_on                   1335
stuck_appliance_off                  1320
appliance_unusual_hours              1170
high_usage_low_occupancy             1155
heating_on_warm_day                   945
weekend_pattern_on_weekday            945
weekday_pattern_on_weekend            915
sustained_overload                    795
gradual_drift_increase                600
multiple_high_power_simultaneous      510
power_spike                           510
impossible_appliance_combo            465
sensor_glitch                         360
gradual_drift_decrease                345
Name: count, dtype: int64

In [15]:
# The raw feature affected by each synthetic anomaly injector.
ANOMALY_SIGNALS = {
    'heating_on_warm_day': ('HPE', 'Heating power'),
    'appliance_unusual_hours': ('WOE', 'Wall-oven power'),
    'high_usage_low_occupancy': ('P', 'Active power'),
    'weekend_pattern_on_weekday': ('CWE', 'Clothes-washer power'),
    'weekday_pattern_on_weekend': ('CWE', 'Clothes-washer power'),
    'stuck_appliance_on': ('DWE', 'Dishwasher power'),
    'stuck_appliance_off': ('HPE', 'Heating power'),
    'gradual_drift_increase': ('B1E', 'Basement circuit power'),
    'gradual_drift_decrease': ('B1E', 'Basement circuit power'),
    'sustained_overload': ('P', 'Active power'),
    'power_spike': ('P', 'Active power'),
    'sensor_glitch': ('V', 'Voltage'),
}

data['multiple_high_power_total'] = data[['HPE', 'DWE', 'CWE']].sum(axis=1)
data['impossible_combo_total'] = data[['HPE', 'WOE']].sum(axis=1)
ANOMALY_SIGNALS.update({
    'multiple_high_power_simultaneous': ('multiple_high_power_total', 'HPE + DWE + CWE power'),
    'impossible_appliance_combo': ('impossible_combo_total', 'HPE + WOE power'),
})


In [16]:
COLORS = {'Normal': '#1f77b4', 'Injected anomaly': '#d62728'}

def plot_anomaly(anomaly_type, context_minutes=180):
    signal, signal_label = ANOMALY_SIGNALS[anomaly_type]
    normal_median = data.loc[data['anomaly_type'].eq('normal'), signal].median()
    anomaly_rows = data.loc[data['anomaly_type'].eq(anomaly_type)].copy()

    # Plot the anomaly sample that differs most from normal, plus three hours of context.
    event_time = anomaly_rows.loc[(anomaly_rows[signal] - normal_median).abs().idxmax(), 'timestamp']
    start = event_time - pd.Timedelta(minutes=context_minutes)
    end = event_time + pd.Timedelta(minutes=context_minutes)
    chart_data = data.loc[
        data['timestamp'].between(start, end) & data['anomaly_type'].isin(['normal', anomaly_type]),
        ['timestamp', signal, 'anomaly_type']
    ].copy()
    chart_data['class'] = np.where(chart_data['anomaly_type'].eq('normal'), 'Normal', 'Injected anomaly')

    fig = px.line(
        chart_data, x='timestamp', y=signal, color='class', markers=True,
        color_discrete_map=COLORS,
        category_orders={'class': ['Normal', 'Injected anomaly']},
        title=f'{anomaly_type.replace("_", " ").title()} — representative event',
        labels={'timestamp': 'Time', signal: signal_label, 'class': ''},
        hover_data={'anomaly_type': True}
    )
    fig.update_traces(marker={'size': 5}, line={'width': 2})
    fig.update_layout(template='plotly_white', height=450)
    fig.show()

for anomaly_type in ANOMALY_SIGNALS:
    if anomaly_type in set(data['anomaly_type']):
        plot_anomaly(anomaly_type)


## Behavior and context feature graphs

For each anomaly type, the next cells select the retained TSFresh or context feature that differs most from normal windows, then show a local line chart.


In [17]:
FEATURES_PATH = Path.cwd() / 'pipeline_cache' / 'ampds_behavior_context_labeled_features.csv'

if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'Cannot find {FEATURES_PATH}')

feature_data = pd.read_csv(FEATURES_PATH, parse_dates=['window_id']).sort_values('window_id')
feature_data['anomaly_type'] = feature_data['anomaly_type'].fillna('normal')
feature_columns = feature_data.select_dtypes(include='number').columns.drop('is_anomaly', errors='ignore')

print(f'Loaded {len(feature_data):,} labeled feature windows and {len(feature_columns):,} numeric features')
display(feature_data['anomaly_type'].value_counts())


Loaded 5,856 labeled feature windows and 830 numeric features


anomaly_type
normal                              5098
stuck_appliance_on                    89
stuck_appliance_off                   88
appliance_unusual_hours               78
high_usage_low_occupancy              77
heating_on_warm_day                   63
weekend_pattern_on_weekday            63
weekday_pattern_on_weekend            61
sustained_overload                    53
gradual_drift_increase                40
multiple_high_power_simultaneous      34
power_spike                           34
impossible_appliance_combo            31
sensor_glitch                         24
gradual_drift_decrease                23
Name: count, dtype: int64

In [18]:
FEATURE_COLORS = {'Normal': '#1f77b4', 'Injected anomaly': '#d62728'}

def most_distinct_feature(anomaly_type):
    normal = feature_data.loc[feature_data['anomaly_type'].eq('normal'), feature_columns]
    anomalous = feature_data.loc[feature_data['anomaly_type'].eq(anomaly_type), feature_columns]
    normal_median = normal.median()
    normal_std = normal.std().replace(0, np.nan)
    separation = (anomalous.median() - normal_median).abs() / normal_std
    return separation.replace([np.inf, -np.inf], np.nan).idxmax()

def plot_feature_anomaly(anomaly_type, context_windows=24):
    feature = most_distinct_feature(anomaly_type)
    normal_median = feature_data.loc[feature_data['anomaly_type'].eq('normal'), feature].median()
    anomalous = feature_data.loc[feature_data['anomaly_type'].eq(anomaly_type)].copy()
    event_time = anomalous.loc[(anomalous[feature] - normal_median).abs().idxmax(), 'window_id']
    start = event_time - pd.Timedelta(minutes=15 * context_windows)
    end = event_time + pd.Timedelta(minutes=15 * context_windows)
    chart_data = feature_data.loc[
        feature_data['window_id'].between(start, end) & feature_data['anomaly_type'].isin(['normal', anomaly_type]),
        ['window_id', feature, 'anomaly_type']
    ].copy()
    chart_data['class'] = np.where(chart_data['anomaly_type'].eq('normal'), 'Normal', 'Injected anomaly')

    fig = px.line(
        chart_data, x='window_id', y=feature, color='class', markers=True,
        color_discrete_map=FEATURE_COLORS,
        category_orders={'class': ['Normal', 'Injected anomaly']},
        title=f'{anomaly_type.replace("_", " ").title()} — {feature}',
        labels={'window_id': 'Window start', feature: 'Feature value', 'class': ''},
        hover_data={'anomaly_type': True}
    )
    fig.update_traces(marker={'size': 5}, line={'width': 2})
    fig.update_layout(template='plotly_white', height=450)
    fig.show()

for anomaly_type in feature_data.loc[feature_data['anomaly_type'].ne('normal'), 'anomaly_type'].unique():
    plot_feature_anomaly(anomaly_type)


## Full dataset feature graphs (Normal vs Anomaly)

Global visualizations across the entire `ampds_behavior_context_labeled_features.csv` dataset, showing normal windows in blue and anomalous windows in red.


In [19]:
# 1. Full dataset timeline: Top distinctive features over time
# Compute features with highest separation between normal and anomaly
normal_features = feature_data.loc[feature_data["is_anomaly"].eq(0), feature_columns]
anomaly_features = feature_data.loc[feature_data["is_anomaly"].eq(1), feature_columns]

separations = (anomaly_features.median() - normal_features.median()).abs() / normal_features.std().replace(0, np.nan)
top_features = separations.sort_values(ascending=False).head(3).index.tolist()

feature_data["status"] = np.where(feature_data["is_anomaly"].eq(1), "Anomaly", "Normal")
STATUS_COLORS = {"Normal": "#1f77b4", "Anomaly": "#d62728"}  # Blue for Normal, Red for Anomaly

for feature in top_features:
    fig = px.scatter(
        feature_data,
        x="window_id",
        y=feature,
        color="status",
        color_discrete_map=STATUS_COLORS,
        category_orders={"status": ["Normal", "Anomaly"]},
        title=f"Full Timeline: {feature} (Normal vs Anomaly)",
        labels={"window_id": "Window Start", feature: feature, "status": "Class"},
        hover_data={"anomaly_type": True, "window_id": True},
        opacity=0.7
    )
    fig.update_traces(marker=dict(size=4))
    fig.update_layout(template="plotly_white", height=450)
    fig.show()


/var/folders/fm/8x7pc1y50zl6zpkn37__j7dr0000gp/T/ipykernel_90630/1255253629.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_data["status"] = np.where(feature_data["is_anomaly"].eq(1), "Anomaly", "Normal")


In [20]:
# 2. Distribution comparison: Box plots for top features
for feature in top_features:
    fig = px.box(
        feature_data,
        x="status",
        y=feature,
        color="status",
        color_discrete_map=STATUS_COLORS,
        category_orders={"status": ["Normal", "Anomaly"]},
        title=f"Distribution: {feature} (Normal vs Anomaly)",
        labels={"status": "Class", feature: feature},
        points="outliers"
    )
    fig.update_layout(template="plotly_white", height=400)
    fig.show()


In [21]:
# 3. Context interaction: Distinctive behavior feature vs outside temperature
if "Temp (C)" in feature_data.columns:
    for feature in top_features[:2]:
        fig = px.scatter(
            feature_data,
            x="Temp (C)",
            y=feature,
            color="status",
            color_discrete_map=STATUS_COLORS,
            category_orders={"status": ["Normal", "Anomaly"]},
            title=f"Behavior vs Context: {feature} vs Outside Temperature",
            labels={"Temp (C)": "Outside Temperature (°C)", feature: feature, "status": "Class"},
            hover_data={"anomaly_type": True, "window_id": True},
            opacity=0.7
        )
        fig.update_traces(marker=dict(size=5))
        fig.update_layout(template="plotly_white", height=450)
        fig.show()


In [22]:
# 4. 2D PCA projection of all features: Normal vs Anomaly in latent space
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = feature_data[feature_columns].fillna(0)
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"])
pca_df["status"] = feature_data["status"].values
pca_df["anomaly_type"] = feature_data["anomaly_type"].values
pca_df["window_id"] = feature_data["window_id"].values

var_ratio = pca.explained_variance_ratio_ * 100

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="status",
    color_discrete_map=STATUS_COLORS,
    category_orders={"status": ["Normal", "Anomaly"]},
    title=f"2D PCA Projection of Feature Space (PC1: {var_ratio[0]:.1f}%, PC2: {var_ratio[1]:.1f}%)",
    labels={"PC1": f"PC1 ({var_ratio[0]:.1f}% var)", "PC2": f"PC2 ({var_ratio[1]:.1f}% var)", "status": "Class"},
    hover_data={"anomaly_type": True, "window_id": True},
    opacity=0.7
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(template="plotly_white", height=500)
fig.show()


In [23]:
# 5. Anomaly counts by type (Normal vs Anomaly breakdown)
type_counts = feature_data["anomaly_type"].value_counts().reset_index()
type_counts.columns = ["anomaly_type", "count"]
type_counts["status"] = np.where(type_counts["anomaly_type"].eq("normal"), "Normal", "Anomaly")

fig = px.bar(
    type_counts,
    x="count",
    y="anomaly_type",
    orientation="h",
    color="status",
    color_discrete_map=STATUS_COLORS,
    category_orders={"status": ["Normal", "Anomaly"]},
    title="Window Counts by Anomaly Type",
    labels={"count": "Number of 15-Minute Windows", "anomaly_type": "Anomaly Type", "status": "Class"}
)
fig.update_layout(template="plotly_white", height=500, yaxis={"categoryorder": "total ascending"})
fig.show()
